In [1]:
import pandas as pd
import duckdb
import numpy as np

con = duckdb.connect()

In [2]:
query = """
SELECT
    customer_id, order_date,
    LAG(order_date) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS prev_order_date,
    
    DATE_DIFF('day',
        LAG(order_date) OVER (
            PARTITION BY customer_id
            ORDER BY order_date
        ),
        order_date
    ) AS gap_days
    
FROM '../../data/transaction/orders.csv'
ORDER BY order_date
"""

In [3]:
df = con.execute(query).df()
print(df.head(10))

   customer_id order_date prev_order_date  gap_days
0       149889 2012-07-04             NaT      <NA>
1        20320 2012-07-04             NaT      <NA>
2        31661 2012-07-04             NaT      <NA>
3       102825 2012-07-04             NaT      <NA>
4       138024 2012-07-04             NaT      <NA>
5        59453 2012-07-04             NaT      <NA>
6         3666 2012-07-04             NaT      <NA>
7       121841 2012-07-04             NaT      <NA>
8       103055 2012-07-04             NaT      <NA>
9        90215 2012-07-04             NaT      <NA>


In [4]:
# Lọc các đơn hàng đầu tiên
df=df[df['gap_days'].notna()]
print(df.head(10))

     customer_id order_date prev_order_date  gap_days
126       134336 2012-07-04      2012-07-04         0
185       125023 2012-07-05      2012-07-04         1
302       136004 2012-07-06      2012-07-04         2
382       143787 2012-07-07      2012-07-06         1
422       148064 2012-07-07      2012-07-05         2
455        72845 2012-07-08      2012-07-06         2
476        54226 2012-07-08      2012-07-08         0
516       107487 2012-07-09      2012-07-08         1
531       150786 2012-07-09      2012-07-05         4
543        30579 2012-07-09      2012-07-05         4


In [5]:
median_gap = df['gap_days'].median()
print(median_gap)

144.0


In [6]:
print(df["gap_days"].describe())

count      556699.0
mean     285.592509
std      389.691558
min             0.0
25%            46.0
50%           144.0
75%           357.0
max          3785.0
Name: gap_days, dtype: Float64
